# Day 7 - Independent Evaluation on WikiANN English

This notebook reports the cached WikiANN evaluation metrics for the final three-approach comparison.

Approach roles:
| Approach | Model | Role |
|---|---|---|
| A | DistilBERT-s42 | Lightweight encoder baseline and best OOD encoder in this run |
| B | DeBERTa-s7 | Best in-domain DeBERTa seed |
| C | LLaMA | Prompted local LLM baseline |

Dataset: `wikiann/en` test split.
WikiANN has PER annotations only for this project setup; EMAIL metrics are reported as null.

Cached JSON inputs used by this notebook:
- `predictions/wikiann/data_stats.json`
- `predictions/wikiann/distilbert_results.json`
- `predictions/wikiann/deberta_results.json`
- `predictions/wikiann/llama_results.json`
- `predictions/wikiann/eval_summary.json` or `predictions/wikiann/results.json`


## Setup


In [ ]:
import json
import pathlib

def find_project_root(start=None):
    current = pathlib.Path(start or pathlib.Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'predictions' / 'wikiann').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root containing predictions/wikiann')


ROOT = find_project_root()
RESULTS_DIR = ROOT / 'predictions' / 'wikiann'
SUMMARY_CANDIDATES = [
    RESULTS_DIR / 'eval_summary.json',
    RESULTS_DIR / 'results.json',
]

DATA_STATS_PATH = RESULTS_DIR / 'data_stats.json'
DISTILBERT_RESULTS_PATH = RESULTS_DIR / 'distilbert_results.json'
DEBERTA_RESULTS_PATH = RESULTS_DIR / 'deberta_results.json'
LLAMA_RESULTS_PATH = RESULTS_DIR / 'llama_results.json'

print('Results:', RESULTS_DIR)


## Load Dataset Summary


In [ ]:
def load_json(path):
    path = pathlib.Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing required JSON file: {path}')
    return json.loads(path.read_text(encoding='utf-8'))


def load_optional_summary():
    for path in SUMMARY_CANDIDATES:
        if path.exists():
            return json.loads(path.read_text(encoding='utf-8'))
    return {}


summary_cache = load_optional_summary()
if DATA_STATS_PATH.exists():
    stats = load_json(DATA_STATS_PATH)
else:
    stats = {
        'dataset': summary_cache.get('dataset', 'wikiann/en'),
        'n_sentences': summary_cache.get('n_sentences'),
        'email_span_count': 0,
        'email_metrics_note': 'Dataset contains no EMAIL spans; EMAIL metrics are reported as null.',
    }

print('Dataset:', stats.get('dataset', 'wikiann/en'))
print('Sentences:', stats.get('n_sentences'))
print('Tokens:', stats.get('n_tokens'))
print('PER spans:', stats.get('per_span_count'))
print('EMAIL spans:', stats.get('email_span_count'))
if stats.get('email_metrics_note'):
    print(stats['email_metrics_note'])


## Load Cached Model Metrics


In [ ]:
def load_model_metrics(model_key, path):
    path = pathlib.Path(path)
    if path.exists():
        metrics = load_json(path)
    else:
        metrics = summary_cache.get(model_key)
        if metrics is None:
            raise FileNotFoundError(
                f'Missing metrics for {model_key}: expected {path} or an entry in eval_summary/results JSON.'
            )
    metrics = dict(metrics)
    if stats.get('email_span_count', 0) == 0 and 'per_f1' in metrics:
        metrics['overall_f1'] = metrics['per_f1']
    return metrics


# Approach A - DistilBERT-s42
approach_a_metrics = load_model_metrics('distilbert', DISTILBERT_RESULTS_PATH)

# Approach B - DeBERTa-s7
approach_b_metrics = load_model_metrics('deberta', DEBERTA_RESULTS_PATH)

# Approach C - LLaMA
approach_c_metrics = load_model_metrics('llama', LLAMA_RESULTS_PATH)

print('Approach A overall_f1:', approach_a_metrics['overall_f1'])
print('Approach B overall_f1:', approach_b_metrics['overall_f1'])
print('Approach C overall_f1:', approach_c_metrics['overall_f1'])


## Consolidated Results


In [ ]:
summary = {
    'dataset': stats.get('dataset', 'wikiann/en'),
    'n_sentences': stats.get('n_sentences'),
    'approaches': {
        'A': 'distilbert',
        'B': 'deberta',
        'C': 'llama',
    },
    'distilbert': approach_a_metrics,
    'deberta': approach_b_metrics,
    'llama': approach_c_metrics,
}
summary


In [ ]:
rows = []
for approach, model_name, metrics in [
    ('A', 'DistilBERT-s42', approach_a_metrics),
    ('B', 'DeBERTa-s7', approach_b_metrics),
    ('C', 'LLaMA', approach_c_metrics),
]:
    rows.append({
        'approach': approach,
        'model': model_name,
        'per_f1': metrics.get('per_f1'),
        'email_f1': metrics.get('email_f1'),
        'overall_f1': metrics.get('overall_f1'),
        'fpr': metrics.get('fpr'),
        'fnr': metrics.get('fnr'),
        'redaction_leak_rate': metrics.get('redaction_leak_rate'),
    })

try:
    import pandas as pd
    try:
        display(pd.DataFrame(rows).set_index(['approach', 'model']))
    except NameError:
        print(pd.DataFrame(rows).set_index(['approach', 'model']))
except ImportError:
    rows


## Findings

Approach A has the strongest WikiANN PER F1 among the three cached runs. Approach C has the lowest leak rate on WikiANN, but it does so with a much higher token false-positive rate. For production masking, Approach A is the preferred default from this run because it gives the best OOD F1 while keeping false positives near encoder levels.


## Bootstrap Significance Test

Skipped for this cached-JSON review. Bootstrap requires per-sentence predictions, while this notebook intentionally reads only aggregate WikiANN JSON metrics.
